# 01 — EDA: Dataset Overview

**Purpose**: Understand the shape, coverage, null profile, and distributions of `data/historical_dataset_clean.parquet`.

Run from the repo root: `jupyter notebook notebooks/01_eda_dataset.ipynb`

In [ ]:
import sys, warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings('ignore')
ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

df = pd.read_parquet(ROOT / 'data' / 'historical_dataset_clean.parquet')
print(f'Shape: {df.shape}')
print(f'Columns: {df.shape[1]}')
print(f'Date range: {df["fiscal_year"].min()} – {df["fiscal_year"].max()}')

## 1 — Row counts by market and year

In [ ]:
by_market = df.groupby('market').size().sort_values(ascending=False)
print('Rows per market:\n', by_market.to_string())

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

by_market.plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Rows per market')
axes[0].set_ylabel('Row count')
axes[0].tick_params(axis='x', rotation=45)

df.groupby(['fiscal_year', 'market']).size().unstack(fill_value=0).plot(
    ax=axes[1], linewidth=1.5
)
axes[1].set_title('Company-year observations over time')
axes[1].set_ylabel('Count')
plt.tight_layout()
plt.show()

## 2 — Period type split

In [ ]:
print(df['period_type'].value_counts())
annual = df[df['period_type'] == 'annual']
print(f'\nAnnual rows: {len(annual):,}  ({len(annual)/len(df)*100:.1f}%)')

## 3 — Null profile (annual rows only)

In [ ]:
null_frac = annual.isna().mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(14, 5))
null_frac.plot(ax=ax, linewidth=0.8, color='crimson')
ax.axhline(0.50, linestyle='--', color='orange', label='50% null')
ax.axhline(0.90, linestyle='--', color='red', label='90% null')
ax.set_title('Null fraction per column (annual rows, sorted descending)')
ax.set_ylabel('Null fraction')
ax.set_xlabel('Column rank')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Columns >50% null: {(null_frac > 0.50).sum()}')
print(f'Columns >90% null: {(null_frac > 0.90).sum()}')
print(f'\nTop 20 most null columns:')
print(null_frac.head(20).to_string())

## 4 — Target variable coverage (forward returns)

In [ ]:
targets = ['forward_return_1y', 'forward_return_3y', 'forward_return_5y',
           'beat_local_market_1y', 'beat_local_market_3y', 'beat_local_market_5y']

print('Target coverage in annual rows:')
for t in targets:
    if t in annual.columns:
        n = annual[t].notna().sum()
        print(f'  {t:<30} {n:>7,}  ({n/len(annual)*100:.1f}%)')

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for i, ret in enumerate(['forward_return_1y', 'forward_return_3y', 'forward_return_5y']):
    if ret in annual.columns:
        s = annual[ret].dropna()
        axes[i].hist(s, bins=80, color='steelblue', edgecolor='white', linewidth=0.3)
        axes[i].set_title(f'{ret}  (n={len(s):,})')
        axes[i].set_xlabel('Return')
plt.tight_layout()
plt.show()

## 5 — ML score distributions

In [ ]:
ml_cols = [c for c in annual.columns if c.startswith('ml_')]
alpha_cols = [c for c in annual.columns if c.startswith('alpha_')]

all_score_cols = ml_cols + alpha_cols
if all_score_cols:
    fig, axes = plt.subplots(2, max(len(ml_cols), len(alpha_cols)), figsize=(20, 8))
    for i, col in enumerate(ml_cols):
        s = annual[col].dropna()
        axes[0, i].hist(s, bins=60, color='navy', edgecolor='white', linewidth=0.3)
        axes[0, i].set_title(col)
    for i, col in enumerate(alpha_cols):
        s = annual[col].dropna()
        axes[1, i].hist(s, bins=60, color='darkorange', edgecolor='white', linewidth=0.3)
        axes[1, i].set_title(col)
    for ax in axes.flat:
        ax.set_visible(False)
    for i, col in enumerate(ml_cols):
        axes[0, i].set_visible(True)
    for i, col in enumerate(alpha_cols):
        axes[1, i].set_visible(True)
    plt.suptitle('ML scores (row 1) and Alpha factor scores (row 2)', fontsize=13)
    plt.tight_layout()
    plt.show()

## 6 — Size category and market coverage

In [ ]:
if 'size_category' in annual.columns:
    print('Size category distribution:')
    print(annual['size_category'].value_counts(dropna=False))
    if 'size_category_imputed' in annual.columns:
        n_imp = annual['size_category_imputed'].sum()
        print(f'\nsize_category_imputed rows: {n_imp:,}  ({n_imp/len(annual)*100:.1f}%)')